# Aralin 09 - Padron ng Disenyo sa Metakognisyon


## Pagsasaayos

Ipinapakita ng notebook na ito ang disenyo ng Metacognition gamit ang Microsoft Agent Framework.

**Mga Kinakailangan:**
- Azure OpenAI deployment na naka-configure sa pamamagitan ng mga environment variable
- Azure CLI na naka-authenticate (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ano ang Metacognition?

Ang Metacognition ay **pag-iisip tungkol sa pag-iisip**. Sa konteksto ng mga AI agent, nangangahulugan ito ng pagbuo ng mga agent na kayang:

- **Magmuni-muni** sa kanilang sariling mga output at proseso ng pangangatwiran
- **Matukoy ang mga pagkakamali** at makabawi nang maayos sa halip na tahimik na mabigo
- **Suriin** kung kumpleto at kapaki-pakinabang ang kanilang mga tugon
- **Iangkop** ang kanilang estratehiya kapag hindi gumana ang unang lapit (hal., magbalik sa backup na sistema)

Ang isang metacognitive na agent ay hindi lamang sumasagot ng mga tanong — binabantayan nito ang sariling pagganap at inaangkop nang mabilis.


## Pangunahing at Panghaliling Mga Tool

Isang karaniwang pattern ng metakognisyon ay ang **estratehiya ng fallback**. Sinusubukan ng ahente ang pangunahing tool muna; kung ito ay mabigo (hal., isang 404 error), nakakikilala ang ahente sa pagkabigo at malinaw na lumilipat sa panghaliling tool.

Ito ay sumasalamin sa mga tunay na sistema kung saan maaaring hindi magamit ang pangunahing mga serbisyo at kailangang tuklasin at suriin ng mga ahente ang isyu bago pumili ng alternatibong landas.

Sa ibaba ay inilalarawan namin ang dalawang tool para sa paghahanap ng flight:
- **Pangunahing** — sumasaklaw sa Paris, Tokyo, at Barcelona
- **Panghalili** — sumasaklaw sa Berlin, Sydney, at New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Ahenteng Nagmumuni-muni sa Sarili na may Pagbawi sa Pagkakamali

Ang ahente sa ibaba ay inutusan na subukan muna ang pangunahing sistema ng paglipad, kilalanin ang mga pagkabigo, at malinaw na lumipat pabalik sa backup na sistema. Pagkatapos ng bawat tugon, maikli nitong sinusuri ang sarili kung ganap nitong nasagot ang tanong ng gumagamit.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Paraan ng Sariling Pagtatasa

Isa pang aspeto ng metakognisyon ay ang **sariling pagtatasa**: isang hiwalay na ahente (o ang parehong ahente sa isang pangalawang pagdaan) ang sinusuri ang isang tugon para sa pagiging kumpleto, katumpakan, at pagiging kapaki-pakinabang.

Sa ibaba, gagawa kami ng isang ahenteng `ResponseEvaluator` na nagbibigay ng iskor sa mga tugon ng ahente ng paglalakbay sa tatlong dimensyon.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Summary

In this lesson you learned how to build **metacognitive agents** using the Microsoft Agent Framework:

- **Pagmumuni-muni sa sarili**: Mga ahenteng nagbabantay sa kanilang sariling pangangatwiran at malinaw na ipinapaalam kung ano ang nangyari.
- **Pagbawi sa error gamit ang mga fallback**: Isang pattern ng pangunahing + backup na kasangkapan kung saan natutukoy ng ahente ang mga pagkabigo (hal., 404 errors) at awtomatikong sumusubok ng alternatibong pinagmulan.
- **Pagsusuri sa sarili**: Isang hiwalay na ahenteng tagasuri na nagbibigay ng puntos sa mga sagot batay sa kabuuan, katumpakan, at pagiging kapaki-pakinabang.

These patterns make agents more robust, transparent, and trustworthy — critical qualities for production deployments.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Paunawa:
Isinaling-wika ang dokumentong ito gamit ang serbisyong pagsasalin ng AI na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagaman nagsusumikap kaming maging tumpak, pakitandaan na maaaring may mga pagkakamali o hindi pagkakatugma ang awtomatikong pagsasalin. Ang orihinal na dokumento sa katutubong wika nito ang dapat ituring na opisyal na sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasaling-tao. Hindi kami mananagot sa anumang hindi pagkakaintindihan o maling interpretasyon na magmumula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
